# 8-Puzzle Game trong Jupyter Notebook

Chạy cell bên dưới để chơi trò 8-puzzle. Có giao diện nút bấm nếu môi trường có `ipywidgets`; nếu không, chương trình sẽ chuyển sang chế độ nhập lệnh bằng console.


In [5]:
import random
import time
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from html import escape


# =========================================================
# 1. PEAS CHO BÀI TOÁN 8 PUZZLE
# =========================================================

PEAS = {
    "P - Performance Measure": [
        "Đưa puzzle về đúng trạng thái đích.",
        "Số ô đúng vị trí càng nhiều càng tốt.",
        "Số bước di chuyển càng ít càng tốt."
    ],
    "E - Environment": [
        "Bàn cờ 3x3.",
        "Có 8 ô số từ 1 đến 8.",
        "Có 1 ô trống ký hiệu là 0 hoặc Blank."
    ],
    "A - Actuators": [
        "UP: di chuyển ô trống lên.",
        "DOWN: di chuyển ô trống xuống.",
        "LEFT: di chuyển ô trống sang trái.",
        "RIGHT: di chuyển ô trống sang phải."
    ],
    "S - Sensors": [
        "Nhìn được trạng thái hiện tại của ma trận 3x3.",
        "Biết vị trí ô trống.",
        "Biết các ô số đang nằm ở đâu."
    ]
}


# =========================================================
# 2. CẤU HÌNH 8 PUZZLE
# =========================================================

GOAL_STATE = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

# Trạng thái ban đầu được chọn để Simple Agent giải được rõ ràng
START_STATE = (
    (1, 2, 3),
    (4, 5, 6),
    (0, 7, 8)
)

ROWS = 3
COLS = 3
SEED = 680
MAX_STEPS = 20

current_state = START_STATE
step_count = 0
move_history = []
rng = random.Random(SEED)

output = widgets.Output()


# =========================================================
# 3. HÀM XỬ LÝ STATE
# =========================================================

def state_to_list(state):
    return [list(row) for row in state]


def list_to_state(board):
    return tuple(tuple(row) for row in board)


def state_to_text(state):
    result = ""
    for row in state:
        result += " ".join("_" if x == 0 else str(x) for x in row) + "\n"
    return result


def find_blank(state):
    """
    Tìm vị trí ô trống 0.
    """
    for r in range(ROWS):
        for c in range(COLS):
            if state[r][c] == 0:
                return r, c


def is_goal(state):
    """
    Kiểm tra state hiện tại có phải goal không.
    """
    return state == GOAL_STATE


def get_valid_actions(state):
    """
    Lấy các action hợp lệ của ô trống.
    """
    r, c = find_blank(state)
    actions = []

    if r > 0:
        actions.append("UP")

    if r < ROWS - 1:
        actions.append("DOWN")

    if c > 0:
        actions.append("LEFT")

    if c < COLS - 1:
        actions.append("RIGHT")

    return actions


def move_blank(state, action):
    """
    Di chuyển ô trống 0 theo action.
    Action ở đây là hướng di chuyển của Blank.
    """
    r, c = find_blank(state)

    dr, dc = 0, 0

    if action == "UP":
        dr, dc = -1, 0
    elif action == "DOWN":
        dr, dc = 1, 0
    elif action == "LEFT":
        dr, dc = 0, -1
    elif action == "RIGHT":
        dr, dc = 0, 1
    else:
        return None

    new_r = r + dr
    new_c = c + dc

    if not (0 <= new_r < ROWS and 0 <= new_c < COLS):
        return None

    board = state_to_list(state)

    # Đổi chỗ ô trống với ô bên cạnh
    board[r][c], board[new_r][new_c] = board[new_r][new_c], board[r][c]

    return list_to_state(board)


def correct_tiles_count(state):
    """
    B(state) = số ô đúng vị trí so với GOAL_STATE.
    Không tính ô trống 0.
    """
    count = 0

    for r in range(ROWS):
        for c in range(COLS):
            if state[r][c] != 0 and state[r][c] == GOAL_STATE[r][c]:
                count += 1

    return count


def get_blank_position_text(state):
    r, c = find_blank(state)
    return f"({r},{c})"


# =========================================================
# 4. SIMPLE AGENT CHO 8 PUZZLE
# =========================================================

class SimplePuzzleAgent:
    def __init__(self, rng):
        self.rng = rng

    def choose_action(self, state):
        """
        Simple Agent hoạt động theo rule:

        IF state == GOAL_STATE
        THEN action = STOP

        ELSE:
            - Lấy các action hợp lệ.
            - Thử từng action.
            - Tính B = số ô đúng vị trí sau khi đi action đó.
            - Chọn action có B cao nhất.
            - Nếu nhiều action bằng điểm nhau thì chọn random.
        """

        if is_goal(state):
            return "STOP", "IF state == GOAL_STATE THEN action = STOP", []

        valid_actions = get_valid_actions(state)
        action_scores = []

        for action in valid_actions:
            next_state = move_blank(state, action)
            score = correct_tiles_count(next_state)

            action_scores.append({
                "action": action,
                "next_state": next_state,
                "score": score
            })

        max_score = max(item["score"] for item in action_scores)

        best_actions = [
            item for item in action_scores
            if item["score"] == max_score
        ]

        chosen = self.rng.choice(best_actions)

        rule = """
IF state != GOAL_STATE
THEN:
    1. Sinh các action hợp lệ của Blank
    2. Với mỗi action, tạo state mới
    3. Tính B(state mới) = số ô đúng vị trí
    4. Chọn action có B lớn nhất
"""

        return chosen["action"], rule, action_scores


agent = SimplePuzzleAgent(rng)


# =========================================================
# 5. GIAO DIỆN HTML GRID
# =========================================================

def puzzle_html(state):
    html = """
    <style>
        .puzzle-wrapper {
            font-family: Arial, sans-serif;
            margin-top: 10px;
        }

        .puzzle-title {
            font-size: 24px;
            font-weight: bold;
            margin-bottom: 12px;
        }

        .puzzle-grid {
            display: grid;
            grid-template-columns: repeat(3, 110px);
            grid-template-rows: repeat(3, 110px);
            gap: 6px;
            margin-bottom: 16px;
        }

        .tile {
            width: 110px;
            height: 110px;
            border: 2px solid #222;
            border-radius: 12px;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            box-sizing: border-box;
        }

        .number-tile {
            background: #90caf9;
        }

        .blank-tile {
            background: #e0e0e0;
        }

        .tile-number {
            font-size: 34px;
            font-weight: bold;
            line-height: 1;
        }

        .tile-blank {
            font-size: 18px;
            font-weight: bold;
            color: #333;
            line-height: 1;
        }

        .tile-position {
            margin-top: 12px;
            font-size: 13px;
            color: #333;
        }

        .info-box {
            background: #f7f7f7;
            color: #111;
            border-left: 5px solid #2196f3;
            padding: 12px;
            border-radius: 8px;
            white-space: pre-wrap;
            font-family: Consolas, monospace;
            font-size: 14px;
            max-width: 780px;
        }

        .history-box {
            background: #fff8e1;
            color: #111;
            border-left: 5px solid #ff9800;
            padding: 12px;
            border-radius: 8px;
            white-space: pre-wrap;
            font-family: Consolas, monospace;
            font-size: 14px;
            max-width: 780px;
            margin-top: 10px;
        }

        .peas-box {
            background: #e8f5e9;
            color: #111;
            border-left: 5px solid #4caf50;
            padding: 12px;
            border-radius: 8px;
            white-space: pre-wrap;
            font-family: Consolas, monospace;
            font-size: 14px;
            max-width: 780px;
            margin-top: 10px;
        }
    </style>

    <div class="puzzle-wrapper">
        <div class="puzzle-title">8 Puzzle - Simple Agent</div>
        <div class="puzzle-grid">
    """

    for r in range(ROWS):
        for c in range(COLS):
            value = state[r][c]

            if value == 0:
                html += f"""
                <div class="tile blank-tile">
                    <div class="tile-blank">Blank</div>
                    <div class="tile-position">({r},{c})</div>
                </div>
                """
            else:
                html += f"""
                <div class="tile number-tile">
                    <div class="tile-number">{value}</div>
                    <div class="tile-position">({r},{c})</div>
                </div>
                """

    html += """
        </div>
    </div>
    """

    return html


def peas_text():
    text = "PEAS CHO AGENT GIẢI 8 PUZZLE\n\n"

    for key, values in PEAS.items():
        text += key + ":\n"
        for item in values:
            text += "- " + item + "\n"
        text += "\n"

    return text


def action_scores_text(action_scores):
    if not action_scores:
        return ""

    text = "BẢNG ĐÁNH GIÁ ACTION THEO SIMPLE AGENT:\n\n"

    for item in action_scores:
        action = item["action"]
        score = item["score"]
        next_state = item["next_state"]

        text += f"Action thử: Blank move {action}\n"
        text += f"B(state sau action) = {score} / 8\n"
        text += "State sau action:\n"
        text += state_to_text(next_state)
        text += "\n"

    return text


# =========================================================
# 6. HÀM HIỂN THỊ
# =========================================================

def render(info_text="", action_scores=None, show_peas=True):
    with output:
        clear_output(wait=True)

        display(HTML(puzzle_html(current_state)))

        basic_info = f"""
{info_text}

TRẠNG THÁI BAN ĐẦU:
{state_to_text(START_STATE)}

TRẠNG THÁI HIỆN TẠI:
{state_to_text(current_state)}

GOAL STATE:
{state_to_text(GOAL_STATE)}
"""

        display(HTML(f"<div class='info-box'>{escape(basic_info)}</div>"))

        if action_scores is not None:
            score_text = action_scores_text(action_scores)
            display(HTML(f"<div class='info-box'>{escape(score_text)}</div>"))

        if len(move_history) == 0:
            history_text = "LỊCH SỬ ACTION:\nChưa có action nào."
        else:
            history_text = "LỊCH SỬ ACTION:\n" + "\n".join(move_history[-15:])

        display(HTML(f"<div class='history-box'>{escape(history_text)}</div>"))

        if show_peas:
            display(HTML(f"<div class='peas-box'>{escape(peas_text())}</div>"))


# =========================================================
# 7. CHƠI THỦ CÔNG
# =========================================================

def manual_move(action):
    global current_state, step_count

    if is_goal(current_state):
        render("Puzzle đã về trạng thái đích.")
        return

    old_state = current_state
    new_state = move_blank(current_state, action)

    if new_state is None:
        render(f"Action {action} không hợp lệ ở state hiện tại.")
        return

    current_state = new_state
    step_count += 1

    old_b = correct_tiles_count(old_state)
    new_b = correct_tiles_count(current_state)

    move_history.append(
        f"Manual Step {step_count} | Blank move: {action} | B: {old_b}/8 -> {new_b}/8"
    )

    info = f"""
MANUAL MOVE

Người chơi chọn action:
Blank move {action}

State trước:
{state_to_text(old_state)}

State sau:
{state_to_text(current_state)}
"""

    if is_goal(current_state):
        info += "\nOUTPUT CUỐI CÙNG: PUZZLE ĐÃ VỀ GOAL."

    render(info)


def on_up_clicked(button):
    manual_move("UP")


def on_down_clicked(button):
    manual_move("DOWN")


def on_left_clicked(button):
    manual_move("LEFT")


def on_right_clicked(button):
    manual_move("RIGHT")


# =========================================================
# 8. SIMPLE AGENT CHẠY 1 BƯỚC
# =========================================================

def simple_agent_one_step(button=None):
    global current_state, step_count

    if is_goal(current_state):
        render("Puzzle đã đạt GOAL. Simple Agent hoàn thành nhiệm vụ.")
        return

    if step_count >= MAX_STEPS:
        render("Đã đạt MAX_STEPS. Simple Agent dừng để tránh lặp quá lâu.")
        return

    old_state = current_state
    old_blank = get_blank_position_text(old_state)
    old_score = correct_tiles_count(old_state)

    action, rule, action_scores = agent.choose_action(current_state)

    if action == "STOP":
        render("Simple Agent chọn STOP vì đã đạt GOAL.", action_scores)
        return

    valid_actions = get_valid_actions(old_state)

    current_state = move_blank(current_state, action)

    new_blank = get_blank_position_text(current_state)
    new_score = correct_tiles_count(current_state)

    step_count += 1

    move_history.append(
        f"Simple Agent Step {step_count} | Blank move: {action} | Blank: {old_blank} -> {new_blank} | B: {old_score}/8 -> {new_score}/8 | Chọn vì B cao nhất"
    )

    info = f"""
SIMPLE AGENT STEP {step_count}

Percept agent nhận được:
Toàn bộ state hiện tại của puzzle.

State trước:
{state_to_text(old_state)}

Vị trí Blank trước:
{old_blank}

Rule được áp dụng:
{rule}

Các action hợp lệ của Blank:
{valid_actions}

Action Simple Agent chọn:
Blank move {action}

Lý do chọn:
Sau khi thử các action hợp lệ, action này cho B(state sau action) cao nhất.

State sau:
{state_to_text(current_state)}

Vị trí Blank sau:
{new_blank}

B trước: {old_score}/8
B sau: {new_score}/8
"""

    if is_goal(current_state):
        info += "\nOUTPUT CUỐI CÙNG: PUZZLE ĐÃ VỀ GOAL."

    render(info, action_scores)


# =========================================================
# 9. SIMPLE AGENT CHẠY TỰ ĐỘNG
# =========================================================

def simple_agent_auto_run(button=None):
    while not is_goal(current_state) and step_count < MAX_STEPS:
        simple_agent_one_step()
        time.sleep(0.5)

    if is_goal(current_state):
        render("OUTPUT CUỐI CÙNG: PUZZLE ĐÃ VỀ GOAL.")
    else:
        render("Simple Agent đã dừng vì đạt MAX_STEPS. Có thể bị kẹt do đây là agent đơn giản.")


# =========================================================
# 10. RESET
# =========================================================

def reset_puzzle(button=None):
    global current_state, step_count, move_history, rng, agent

    rng = random.Random(SEED)
    agent = SimplePuzzleAgent(rng)

    current_state = START_STATE
    step_count = 0
    move_history = []

    render("Đã reset puzzle về trạng thái ban đầu.")


# =========================================================
# 11. BUTTON GIAO DIỆN
# =========================================================

up_button = widgets.Button(
    description="UP",
    button_style="",
    layout=widgets.Layout(width="90px")
)

down_button = widgets.Button(
    description="DOWN",
    button_style="",
    layout=widgets.Layout(width="90px")
)

left_button = widgets.Button(
    description="LEFT",
    button_style="",
    layout=widgets.Layout(width="90px")
)

right_button = widgets.Button(
    description="RIGHT",
    button_style="",
    layout=widgets.Layout(width="90px")
)

agent_step_button = widgets.Button(
    description="Simple Agent 1 bước",
    button_style="success",
    layout=widgets.Layout(width="180px")
)

agent_auto_button = widgets.Button(
    description="Simple Agent tự chạy",
    button_style="info",
    layout=widgets.Layout(width="190px")
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning",
    layout=widgets.Layout(width="110px")
)

up_button.on_click(on_up_clicked)
down_button.on_click(on_down_clicked)
left_button.on_click(on_left_clicked)
right_button.on_click(on_right_clicked)

agent_step_button.on_click(simple_agent_one_step)
agent_auto_button.on_click(simple_agent_auto_run)
reset_button.on_click(reset_puzzle)

manual_buttons = widgets.HBox([
    up_button,
    down_button,
    left_button,
    right_button
])

agent_buttons = widgets.HBox([
    agent_step_button,
    agent_auto_button,
    reset_button
])


# =========================================================
# 12. HIỂN THỊ CHƯƠNG TRÌNH
# =========================================================

title = widgets.HTML(
    value="""
    <h2 style='font-family: Arial; margin-bottom: 5px;'>
        8 Puzzle - Simple Agent
    </h2>
    <p style='font-family: Arial; font-size: 14px;'>
        Bấm các nút bên dưới để chơi thủ công hoặc cho Simple Agent chạy.
    </p>
    """
)

manual_label = widgets.HTML(
    value="<b style='font-family: Arial;'>Điều khiển thủ công:</b>"
)

agent_label = widgets.HTML(
    value="<b style='font-family: Arial;'>Điều khiển Simple Agent:</b>"
)

ui = widgets.VBox([
    title,
    manual_label,
    manual_buttons,
    agent_label,
    agent_buttons,
    output
])

display(ui)

render("""
MÔI TRƯỜNG 8 PUZZLE BAN ĐẦU

Quy ước:
- Ô xanh là ô số.
- Ô xám Blank là ô trống, tương ứng số 0.
- Goal State là:
  1 2 3
  4 5 6
  7 8 _

Simple Agent dùng rule:
IF state == GOAL_STATE THEN STOP
ELSE chọn action làm B lớn nhất

Trong đó:
B = số ô đúng vị trí so với goal.

Lưu ý:
Action RIGHT nghĩa là Blank di chuyển sang phải.
Action LEFT nghĩa là Blank di chuyển sang trái.
Action UP nghĩa là Blank di chuyển lên.
Action DOWN nghĩa là Blank di chuyển xuống.

Bấm 'Simple Agent 1 bước' để xem từng bước.
Bấm 'Simple Agent tự chạy' để chạy đến goal.
""")